# Beyond Shortcuts: Multimodal Fake News Detection
## Notebook 02: Frozen CLIP (ViT-B/32) Feature Extraction (Phase A2)

### Purpose
Extract and cache 512-dimensional, L2-normalized `float16` embeddings for both headline text and news images using `openai/clip-vit-base-patch32`.


In [ ]:
# 1. Environment & GPU Check
import torch
print('PyTorch version:', torch.__version__)
if torch.cuda.is_available():
    print('GPU Available:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM: {vram:.1f} GB')
else:
    print('[WARNING] CUDA not detected! Go to right sidebar -> Notebook options -> Accelerator -> GPU T4.')


In [ ]:
# 2. Dataset Definition & Smart Path Resolver
import os
import time
from typing import List, Tuple

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor

def find_input_paths() -> Tuple[str, str]:
    tables = []
    image_dirs = []
    for base in ['/kaggle/input', '/kaggle/working', '.']:
        if not os.path.exists(base):
            continue
        for root, dirs, files in os.walk(base):
            if 'table.parquet' in files:
                tables.append(os.path.join(root, 'table.parquet'))
            if 'images_v1' in dirs:
                image_dirs.append(os.path.join(root, 'images_v1'))
                
    print(f'Detected Tables: {tables}')
    print(f'Detected Image Dirs: {image_dirs}')
    
    chosen_table = None
    chosen_img_dir = None
    
    if image_dirs:
        for img_dir in image_dirs:
            parent = os.path.dirname(img_dir)
            candidate = os.path.join(parent, 'table.parquet')
            if os.path.exists(candidate):
                chosen_table = candidate
                chosen_img_dir = img_dir
                break
        if chosen_img_dir is None:
            chosen_img_dir = image_dirs[0]
            
    if chosen_table is None and tables:
        for t in tables:
            if 'prepare' in t.lower() or 'n1' in t.lower():
                chosen_table = t
                break
        if chosen_table is None:
            chosen_table = tables[0]
            
    print(f'Selected table: {chosen_table}')
    print(f'Selected images: {chosen_img_dir}')
    return chosen_table, chosen_img_dir

class MultimodalCLIPDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_dir: str):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.ids = self.df['id'].astype(str).tolist()
        
        # In our finalized Parquet schema, clean_title was standardized to 'text'
        text_col = 'text' if 'text' in self.df.columns else ('clean_title' if 'clean_title' in self.df.columns else 'title')
        self.texts = [
            str(t) if pd.notna(t) and str(t).strip() else 'news image'
            for t in self.df[text_col]
        ]
        self.img_paths = [os.path.join(self.image_dir, f'{i}.jpg') for i in self.ids]
        self.fallback = Image.new('RGB', (224, 224), color=(128, 128, 128))

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[str, str, Image.Image]:
        item_id = self.ids[idx]
        text = self.texts[idx]
        img_path = self.img_paths[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = self.fallback
        return item_id, text, image

def clip_collate_fn(batch, processor: CLIPProcessor):
    ids = [b[0] for b in batch]
    texts = [b[1] for b in batch]
    images = [b[2] for b in batch]
    inputs = processor(
        text=texts, images=images, return_tensors='pt', padding=True, truncation=True, max_length=77
    )
    return ids, inputs


In [ ]:
# 3. Run Embedding Extraction Pipeline
def to_feature_tensor(output):
    if hasattr(output, 'pooler_output') and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, 'last_hidden_state') and output.last_hidden_state is not None:
        return output.last_hidden_state[:, 0, :]
    return output

def extract_and_save(
    output_dir: str = '/kaggle/working',
    model_name: str = 'openai/clip-vit-base-patch32',
    batch_size: int = 128,
    num_workers: int = 4
):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    table_path, image_dir = find_input_paths()
    print(f'Selected table: {table_path}')
    print(f'Selected images: {image_dir}')
    
    df = pd.read_parquet(table_path)
    n_total = len(df)
    print(f'Processing {n_total:,} rows...')
    
    print(f'Loading CLIP ({model_name})...')
    processor = CLIPProcessor.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(device)
    model.eval()
    
    dataset = MultimodalCLIPDataset(df, image_dir)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=lambda b: clip_collate_fn(b, processor),
        pin_memory=(device == 'cuda'),
    )
    
    all_ids, text_embs, image_embs = [], [], []
    start = time.time()
    
    with torch.no_grad():
        for b_ids, inputs in tqdm(loader, desc='Extracting CLIP'):
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.amp.autocast(device_type='cuda' if device == 'cuda' else 'cpu', dtype=torch.float16):
                t_raw = model.get_text_features(input_ids=inputs['input_ids'], attention_mask=inputs.get('attention_mask'))
                t_feat = to_feature_tensor(t_raw)
                t_feat = t_feat / t_feat.norm(dim=-1, keepdim=True)
                
                i_raw = model.get_image_features(pixel_values=inputs['pixel_values'])
                i_feat = to_feature_tensor(i_raw)
                i_feat = i_feat / i_feat.norm(dim=-1, keepdim=True)
                
            all_ids.extend(b_ids)
            text_embs.append(t_feat.cpu().to(torch.float16).numpy())
            image_embs.append(i_feat.cpu().to(torch.float16).numpy())
            
    elapsed = (time.time() - start) / 60
    print(f'\nExtracted in {elapsed:.2f} minutes!')
    
    ids_arr = np.array(all_ids, dtype=object)
    text_arr = np.concatenate(text_embs, axis=0)
    image_arr = np.concatenate(image_embs, axis=0)
    
    # Integrity assertions
    assert len(ids_arr) == n_total
    assert text_arr.shape == (n_total, 512)
    assert image_arr.shape == (n_total, 512)
    assert not np.isnan(text_arr).any() and not np.isnan(image_arr).any()
    
    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, 'ids.npy'), ids_arr)
    np.save(os.path.join(output_dir, 'text_emb.npy'), text_arr)
    np.save(os.path.join(output_dir, 'image_emb.npy'), image_arr)
    df.to_parquet(os.path.join(output_dir, 'metadata.parquet'), index=False)
    
    print('\nAll files saved successfully to /kaggle/working:')
    print('  - ids.npy:', ids_arr.shape)
    print('  - text_emb.npy:', text_arr.shape, text_arr.dtype)
    print('  - image_emb.npy:', image_arr.shape, image_arr.dtype)
    print('  - metadata.parquet:', len(df), 'rows')

# Execute
extract_and_save()


In [ ]:
# 4. Check Saved Files
!ls -lh /kaggle/working
